# Economic Dispatch — Eval CPS

Menentukan **economic dispatch** 5 generator untuk beban **650 MW** dan **740 MW**.
Metode: **lambda-iteration** (bisection pada incremental cost) dengan **iterasi maks = 50**.

**Model biaya kuadratik:** $C_i(P_i) = a_i P_i^2 + b_i P_i + c_i$ ($/jam).
**Objektif:** minimasi $\sum_i C_i(P_i)$ dengan kendala keseimbangan daya
$\sum_i P_i = P_D$ dan batas $P_i^{min} \le P_i \le P_i^{max}$.

**Syarat optimum (tanpa rugi saluran):** incremental cost sama,
$\frac{dC_i}{dP_i} = 2a_iP_i + b_i = \lambda$, sehingga
$P_i(\lambda) = \mathrm{clip}\!\left(\frac{\lambda - b_i}{2a_i},\, P_i^{min},\, P_i^{max}\right)$.
Cari $\lambda$ agar $\sum_i P_i(\lambda) = P_D$ via **bisection**.

## 1. Data generator

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

# Gen : Pmin, Pmax, a, b, c   (C = a P^2 + b P + c)
gen = {
  'G1': dict(pmin=0, pmax=300, a=0.002, b=10, c=100),
  'G2': dict(pmin=0, pmax=250, a=0.003, b=8,  c=120),
  'G3': dict(pmin=0, pmax=200, a=0.004, b=6,  c=150),
  'G4': dict(pmin=0, pmax=150, a=0.005, b=5,  c=180),
  'G5': dict(pmin=0, pmax=100, a=0.006, b=4,  c=200),
}
names=list(gen); a=np.array([gen[g]['a'] for g in names]); b=np.array([gen[g]['b'] for g in names])
c=np.array([gen[g]['c'] for g in names]); pmin=np.array([gen[g]['pmin'] for g in names],float)
pmax=np.array([gen[g]['pmax'] for g in names],float)
print('Total Pmax =', pmax.sum(), 'MW ; Total Pmin =', pmin.sum(),'MW')
print('Beban yang diuji: 650 MW dan 740 MW (keduanya layak: Pmin<=PD<=Pmax).')

## 2. Fungsi dispatch untuk lambda tertentu + biaya total

In [ ]:
def P_of_lambda(lam):
    return np.clip((lam - b)/(2*a), pmin, pmax)
def total_cost(P):
    return float(np.sum(a*P**2 + b*P + c))
def incremental_cost(P):
    return 2*a*P + b

## 3. Lambda-iteration (bisection, maks 50 iterasi)

Tiap iterasi: ambil $\lambda$ tengah, hitung $\sum P_i(\lambda)$, perkecil selang.
Catat **biaya total** dan **daya tiap generator** di setiap iterasi (untuk grafik &
tabel konvergensi). Nilai dicatat pada dispatch yang **feasible** (daya di-rescale
ringan agar $\sum P_i = P_D$ tepat saat mengevaluasi biaya per-iterasi).

In [ ]:
def economic_dispatch(PD, max_iter=50, tol=1e-6):
    lam_lo = (2*a*pmin + b).min()      # lambda minimum yang masuk akal
    lam_hi = (2*a*pmax + b).max()      # lambda maksimum
    hist = []  # (iter, lambda, cost, mismatch, P-array)
    P = P_of_lambda(0.5*(lam_lo+lam_hi))
    for it in range(1, max_iter+1):
        lam = 0.5*(lam_lo + lam_hi)
        P = P_of_lambda(lam)
        s = P.sum(); mismatch = s - PD
        # untuk pelaporan biaya per-iterasi, evaluasi pada dispatch feasible:
        # geser daya proporsional pada unit yang belum di batas agar sum == PD
        Pf = feasible_dispatch(lam, PD)
        hist.append(dict(iter=it, lam=lam, cost=total_cost(Pf), mismatch=mismatch,
                         P=Pf.copy(), sumP=float(Pf.sum())))
        if abs(mismatch) < tol:
            # tetap lanjut mencatat sampai cukup (konvergen); hentikan bila sangat kecil
            pass
        if mismatch > 0:   # produksi > beban -> turunkan lambda
            lam_hi = lam
        else:              # produksi < beban -> naikkan lambda
            lam_lo = lam
    return hist

def feasible_dispatch(lam, PD):
    # dispatch dari lambda, lalu koreksi kecil agar sum tepat = PD menjaga batas
    P = P_of_lambda(lam); diff = PD - P.sum()
    for _ in range(100):
        if abs(diff) < 1e-9: break
        if diff > 0:  room = pmax - P
        else:         room = P - pmin
        free = room > 1e-12
        if not free.any(): break
        share = room[free]/room[free].sum()
        P[free] += np.sign(diff)*min(abs(diff), room[free].sum())*share
        diff = PD - P.sum()
    return np.clip(P, pmin, pmax)

## 4. Jalankan untuk beban 650 MW dan 740 MW

In [ ]:
results={}
for PD in [650, 740]:
    hist = economic_dispatch(PD, max_iter=50)
    results[PD]=hist
    last=hist[-1]
    print(f'\n===== Beban {PD} MW =====')
    print('lambda akhir  = %.5f $/MWh'%last['lam'])
    print('Total biaya   = %.4f $/jam'%last['cost'])
    print('Sum P         = %.4f MW (target %d)'%(last['sumP'],PD))
    for g,P in zip(names,last['P']):
        print(f'  {g}: {P:8.3f} MW')

## 5. Grafik: biaya pembangkitan vs iterasi (kedua beban)

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4.5))
for ax,PD in zip(axs,[650,740]):
    h=results[PD]; its=[x['iter'] for x in h]; cost=[x['cost'] for x in h]
    ax.plot(its,cost,'-o',ms=3,color='#C44E52')
    ax.set_xlabel('Iterasi'); ax.set_ylabel('Total biaya pembangkitan ($/jam)')
    ax.set_title(f'Konvergensi biaya — beban {PD} MW'); ax.grid(alpha=0.3)
    ax.annotate('%.2f'%cost[-1],(its[-1],cost[-1]),textcoords='offset points',xytext=(-30,8),fontsize=8)
plt.tight_layout(); plt.savefig('ed_cost_convergence.png',dpi=150,bbox_inches='tight'); plt.show()
print('saved ed_cost_convergence.png')

## 6. Grafik: daya tiap generator vs iterasi

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4.5))
colors=['#4C72B0','#DD8452','#55A868','#C44E52','#8172B3']
for ax,PD in zip(axs,[650,740]):
    h=results[PD]; its=[x['iter'] for x in h]
    Parr=np.array([x['P'] for x in h])  # iter x 5
    for k,g in enumerate(names):
        ax.plot(its,Parr[:,k],'-',color=colors[k],label=g)
    ax.set_xlabel('Iterasi'); ax.set_ylabel('Daya (MW)')
    ax.set_title(f'Daya tiap generator — beban {PD} MW'); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig('ed_power_convergence.png',dpi=150,bbox_inches='tight'); plt.show()
print('saved ed_power_convergence.png')

## 7. Tabel hasil akhir (untuk disalin ke LaTeX)

In [ ]:
import pandas as pd
for PD in [650,740]:
    last=results[PD][-1]
    df=pd.DataFrame({'Gen':names,'Pmin':pmin,'Pmax':pmax,'a':a,'b':b,'c':c,
                     'P (MW)':np.round(last['P'],3),
                     'IC=2aP+b':np.round(incremental_cost(last['P']),4),
                     'Biaya ($/jam)':np.round(a*last['P']**2+b*last['P']+c,3)})
    print(f'\n===== Beban {PD} MW | lambda={last["lam"]:.5f} | total={last["cost"]:.3f} $/jam =====')
    print(df.to_string(index=False))